# 卷积神经网络（下）残差连接与 ResNet

本节验证一个经典结论：**网络变深后纯堆叠 Conv 反而难训，残差连接可以化解**。

做法：在 CIFAR-10 小子集上用两个深度相同的网络对比——**Plain-Net**（8 个 Conv 直接堆）vs **ResNet-style**（同样 8 个 Conv，但每 2 个 conv 加一个 skip 连接）。

## 1. CIFAR-10 子集

完整 CIFAR-10 有 50k 训练、10k 测试。为了 notebook 跑得快，只用 5000/1000 子集；演示完结论后，full data + 更多 epoch 才是实际跑法。

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

CACHE = os.path.expanduser('~/.cache/torch_data')
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
train_full = datasets.CIFAR10(CACHE, train=True,  download=True, transform=tfm)
test_full  = datasets.CIFAR10(CACHE, train=False, download=True, transform=tfm)

torch.manual_seed(0)
tr_idx = torch.randperm(len(train_full))[:5000].tolist()
te_idx = torch.randperm(len(test_full))[:1000].tolist()
train_set = Subset(train_full, tr_idx)
test_set  = Subset(test_full,  te_idx)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256)

classes = train_full.classes
print(f'train {len(train_set)}  test {len(test_set)}  classes {classes}')

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
from nndl.runner import RunnerV3


看几张样本：

In [ ]:
x, y = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(14, 2.5))
for ax, img, lab in zip(axes, x[:8], y[:8]):
    img_disp = img * torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1) + \
               torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    ax.imshow(img_disp.permute(1, 2, 0).clamp(0, 1))
    ax.set_title(classes[lab.item()], fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Plain vs Residual block

两种 block 都包含两个 `3×3 Conv + BN + ReLU`；差别只是 ResBlock 加了一条从 block 输入直连到 block 输出的 skip。

In [ ]:
class PlainBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        return x


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        # 当输入/输出通道数 or 尺寸不一致时，shortcut 用 1x1 conv 投影
        self.shortcut = nn.Identity() if (in_ch == out_ch and stride == 1) else \
                        nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                                      nn.BatchNorm2d(out_ch))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))      # ← 加 skip 连接

## 3. 同样深度的两个网络

深度 = stem (1 Conv) + 3 stage × 2 block × 2 conv = 13 Conv 层。Plain / Res 网络只在 block 类型上不同。

In [ ]:
class Net(nn.Module):
    def __init__(self, block, n_class=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16), nn.ReLU(inplace=True),
        )
        self.stage1 = nn.Sequential(block(16, 16, stride=1), block(16, 16, stride=1))
        self.stage2 = nn.Sequential(block(16, 32, stride=2), block(32, 32, stride=1))
        self.stage3 = nn.Sequential(block(32, 64, stride=2), block(64, 64, stride=1))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, n_class)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x); x = self.stage2(x); x = self.stage3(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

for name, block in [('Plain', PlainBlock), ('Res', ResBlock)]:
    n = sum(p.numel() for p in Net(block).parameters())
    print(f'{name}-Net params: {n:,}')

## 4. 训练 + 比较

两个网络用相同的优化器、相同的随机种子、相同的 epoch 数。CPU 上 5 epoch 大约 2-3 分钟一个网络。

In [ ]:
from nndl import accuracy

def train_one(model, epochs=5, lr=0.01):
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    runner = RunnerV3(model, opt, nn.CrossEntropyLoss(),
                      metric_fn=accuracy, higher_is_better=True)
    runner.fit(train_loader, test_loader, num_epochs=epochs, log_every=1)
    return runner.history

print('=== Plain-Net ===')
torch.manual_seed(0)
plain_hist = train_one(Net(PlainBlock))

print('=== ResNet-style ===')
torch.manual_seed(0)
res_hist = train_one(Net(ResBlock))

### 曲线对比

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(plain_hist['train_loss'], '-o', label='Plain')
ax1.plot(res_hist['train_loss'],   '-o', label='ResNet')
ax1.set_xlabel('epoch'); ax1.set_ylabel('train CE loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(plain_hist['dev_metric'], '-o', label='Plain')
ax2.plot(res_hist['dev_metric'],   '-o', label='ResNet')
ax2.set_xlabel('epoch'); ax2.set_ylabel('test accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

残差连接的几个直观解释：

- **梯度路径更短**：skip 让早层梯度可以直接从最后一层回传，缓解梯度消失。
- **学习残差函数 $\mathcal{F}(x) = H(x) - x$ 比直接学习 $H(x)$ 容易**：当真函数接近 identity 时，残差只要学接近 0 的小修正即可。
- 在小数据 + 浅网络（这里 13 层）上差距可能不夸张；越深的网络（ResNet-50/100+）差距越明显。

## 5. `torchvision.models.resnet18` API

torchvision 内置了 ResNet18/34/50/101/152 和预训练权重，实际项目里直接用：

In [ ]:
from torchvision.models import resnet18

# 从零开始训练（CIFAR-10 标签 10 类）
m = resnet18(weights=None, num_classes=10)
n_params = sum(p.numel() for p in m.parameters())
print(f'torchvision resnet18 params: {n_params:,}')

# 注意：原版 ResNet18 接受 224x224 输入；CIFAR-10 32x32 会一直被 stride=2 折掉
# 实战中通常会改第一个 Conv (kernel 7→3, stride 2→1) 并去掉首个 MaxPool
out = m(torch.zeros(1, 3, 224, 224))
print(f'output shape on 224x224 input: {tuple(out.shape)}')

## 小结

- **残差连接 (skip connection)** 让深网络更易优化，是 ResNet/DenseNet/Transformer 等架构的基础。
- **BatchNorm** 几乎总是和现代 Conv 网络配合（`Conv → BN → ReLU` 是标配三件套）。
- 写自己的 ResNet：注意 stride>1 或通道数变化时，shortcut 要用 `1×1 Conv` 投影对齐。
- 实战用 `torchvision.models.resnet18(weights=...)` 加预训练 ImageNet 权重 → 微调，比从零训快得多、效果好得多（chap7 会展开学习率 / 优化器策略）。

## 附：全量数据上的端到端 ResNet18（CIFAR-10，对照书本案例）

前面用 5000/1000 的小子集快速演示了残差连接的效果。配套书《案例与实践（2）》§5.5 给出的是**全量 CIFAR-10**（训练 40000 / 验证 10000 / 测试 10000）上、直接调用 `torchvision.models.resnet18` 的端到端流程。这里把书里的完整版本搬过来。

> **运行说明**：本节依赖 `torchvision`（提供 `resnet18` 与 `Normalize`）——当前 venv 已安装；数据用本地 CIFAR-10 原始批文件，下面的数据 cell 会在 `../../../practice-in-paddle/dataset/cifar-10-batches-py` 等候选位置自动查找，无需联网下载。CPU 上跑满 30 个回合较慢，可调小 `num_epochs`，或对训练集做代表性抽样（如 `Subset(train_dataset, range(0, 40000, 4))`）做一次快跑。

若想换用 `torchvision` 自带下载，也可改成 `torchvision.datasets.CIFAR10(..., download=True)`；本节按书走手写 pickle 读取。

### 1. 数据读取与 Dataset（全量 40000/10000/10000）

按批读取 CIFAR-10：训练集 5 个批文件，前 4 个（40000 条）合并为训练集、第 5 个（10000 条）作验证集，`test_batch`（10000 条）作测试集。图像 reshape 成 `(3,32,32)` 并归一化到 $[0,1]$，再做通道级标准化。

In [ ]:
import os
import pickle
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision.transforms import Normalize       # 需要 torchvision


def load_cifar10_batch(folder_path, batch_id=1, mode='train'):
    if mode == 'test':
        file_path = os.path.join(folder_path, 'test_batch')
    else:
        file_path = os.path.join(folder_path, 'data_batch_' + str(batch_id))
    # 加载数据集文件
    with open(file_path, 'rb') as batch_file:
        batch = pickle.load(batch_file, encoding='latin1')
    imgs = batch['data'].reshape((len(batch['data']), 3, 32, 32)) / 255.
    labels = batch['labels']
    return np.array(imgs, dtype=np.float32), np.array(labels)


class CIFAR10Dataset(torch.utils.data.Dataset):
    def __init__(self, folder_path, mode='train'):
        if mode == 'train':
            # 加载 batch1-batch4 作为训练集（40000 条）
            self.imgs, self.labels = load_cifar10_batch(folder_path=folder_path, batch_id=1, mode='train')
            for i in range(2, 5):
                imgs_batch, labels_batch = load_cifar10_batch(folder_path=folder_path, batch_id=i, mode='train')
                self.imgs = np.concatenate([self.imgs, imgs_batch])
                self.labels = np.concatenate([self.labels, labels_batch])
        elif mode == 'dev':
            # batch5 作验证集（10000 条）
            self.imgs, self.labels = load_cifar10_batch(folder_path=folder_path, batch_id=5, mode='dev')
        elif mode == 'test':
            # test_batch 作测试集（10000 条）
            self.imgs, self.labels = load_cifar10_batch(folder_path=folder_path, mode='test')
        self.transform = Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010])

    def __getitem__(self, idx):
        img = torch.from_numpy(self.imgs[idx])      # (3,32,32)，已归一化到 [0,1]
        label = int(self.labels[idx])
        img = self.transform(img)                   # 通道级标准化
        return img, label

    def __len__(self):
        return len(self.imgs)


torch.manual_seed(100)
# 在若干候选位置里找本地 CIFAR-10 原始批目录（不联网、不下载）：
# data_batch_1..5 / test_batch / batches.meta 都在该目录下。
_CIFAR_CANDIDATES = [
    os.path.join('..', 'datasets', 'cifar-10-batches-py'),
    os.path.join('..', '..', '..', 'practice-in-paddle', 'dataset', 'cifar-10-batches-py'),
    os.path.join('datasets', 'cifar', 'cifar-10-batches-py'),
]
cifar_path = next((p for p in _CIFAR_CANDIDATES
                   if os.path.isfile(os.path.join(p, 'data_batch_1'))), _CIFAR_CANDIDATES[0])
print('CIFAR-10 数据目录：', cifar_path)
train_dataset = CIFAR10Dataset(cifar_path, mode='train')
dev_dataset = CIFAR10Dataset(cifar_path, mode='dev')
test_dataset = CIFAR10Dataset(cifar_path, mode='test')
print(f'train/dev/test = {len(train_dataset)}/{len(dev_dataset)}/{len(test_dataset)}')

### 2. 模型构建：torchvision 自带 ResNet18

直接用 PyTorch 自带的 `resnet18`。它默认面向 ImageNet（$224\times224$、1000 类），实例化时传 `weights=None`（从随机初始化训练）和 `num_classes=10`（CIFAR-10 类别数）。

> 对 $32\times32$ 输入这是粗略沿用；更贴合小尺寸的 ResNet 变体（改第一个 conv 的 kernel/stride、去掉首个 MaxPool）见 `nndl-practice` 仓库，以及本 notebook 上文“自己写 ResNet”那一节。

In [ ]:
from torchvision.models import resnet18           # 需要 torchvision

resnet18_model = resnet18(weights=None, num_classes=10)
n_params = sum(p.numel() for p in resnet18_model.parameters())
print(f'torchvision resnet18 params: {n_params:,}')

### 3. 模型训练（Adam + 权重衰减，30 个回合）

ResNet18 层多、参数大，纯 SGD 难收敛；改用带自适应学习率的 **Adam**，叠加权重衰减抑制过拟合，训练 30 个回合，自动保留验证集准确率最高的快照。`RunnerV3` 把 `device` 一并接管，自动搬迁模型与批数据。

In [ ]:
import sys
import torch.nn.functional as F
import torch.optim as optim

# 从父目录引入 nndl 训练框架
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
from nndl import RunnerV3, accuracy

# 指定运行设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 批大小
batch_size = 64
# 加载数据
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)
# 定义网络
model = resnet18_model
# 定义优化器：Adam + 权重衰减
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.005)
# 定义损失函数与评价指标
loss_fn = F.cross_entropy
metric = accuracy
# 实例化 RunnerV3，把设备交给 Runner 一并管理
runner = RunnerV3(model, optimizer, loss_fn, metric_fn=metric, device=device)
# 模型训练
runner.fit(train_loader, dev_loader, num_epochs=30, log_every=1, best_path="./checkpoint/model_best.pt")

### 损失和准确率变化趋势

训练结束后，`runner.history` 记录了逐回合的训练损失、验证损失与验证准确率（keys：`train_loss` / `dev_loss` / `dev_metric`）。左图叠加训练/验证损失，右图给出验证准确率随回合的变化。

In [ ]:
import matplotlib.pyplot as plt


def plot_loss4(history):
    """按书图“损失和准确率变化趋势”画 RunnerV3.history。

    左图：训练损失 + 验证损失随回合变化；右图：验证准确率随回合变化。
    history 取自 RunnerV3.history（keys：train_loss / dev_loss / dev_metric）。
    """
    train_loss = history["train_loss"]
    dev_loss = history["dev_loss"]
    dev_metric = history["dev_metric"]
    epochs = range(1, len(train_loss) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, train_loss, "-o", ms=3, label="train loss")
    ax1.plot(epochs, dev_loss, "-o", ms=3, label="dev loss")
    ax1.set_xlabel("epoch")
    ax1.set_ylabel("loss")
    ax1.set_title("loss")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(epochs, dev_metric, "-o", ms=3, color="tab:green", label="dev accuracy")
    ax2.set_xlabel("epoch")
    ax2.set_ylabel("accuracy")
    ax2.set_title("accuracy")
    ax2.legend()
    ax2.grid(alpha=0.3)

    fig.tight_layout()
    plt.show()


plot_loss4(runner.history)

### 4. 模型评价与预测

载入验证集上最优的快照，在测试集上做最终评价；再对一条测试样本做预测，对照真实类别。书中参考结果约为 `[Test] accuracy/loss: 0.7094/0.9058`。

In [ ]:
# 加载最优模型
runner.load_model("./checkpoint/model_best.pt")
# 模型评价
score, loss = runner.evaluate(test_loader)
print(f"[Test] accuracy/loss: {score:.4f}/{loss:.4f}")

# 模型预测
id2label = {0: 'airplane', 1: 'automobile', 2: 'bird', 3: 'cat', 4: 'deer',
            5: 'dog', 6: 'frog', 7: 'horse', 8: 'ship', 9: 'truck'}
X, label_ids = next(iter(test_loader))
logits = runner.predict(X)
sample_id = 2
pred_class = id2label[logits[sample_id].argmax().item()]
true_class = id2label[label_ids[sample_id].item()]
print(f"The true category is {true_class} and the predicted category is {pred_class}")

把这条测试样本（`sample_id = 2`）画出来：标准化后的张量先按通道还原回 $[0,1]$ 再显示，标题同时标注预测类别与真实类别。

In [ ]:
def _denormalize(img):
    """把标准化后的 (3,32,32) 张量还原回 [0,1] 以便显示。"""
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std = torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1)
    return (img.detach().cpu() * std + mean).clamp(0, 1)


def plot_test_vis(img, pred_label, true_label):
    """画单张测试图 + 预测/真实类别标题（sample_id=2）。"""
    fig, ax = plt.subplots(figsize=(2.4, 2.6))
    ax.imshow(_denormalize(img).permute(1, 2, 0).numpy())
    ax.set_title(f"pred: {pred_label}\ntrue: {true_label}", fontsize=9)
    ax.axis("off")
    fig.tight_layout()
    plt.show()


plot_test_vis(X[sample_id], pred_class, true_class)